In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [ ]:
df = spark.read.json("data/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

In [ ]:
df.show(10, truncate=False)

In [ ]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

In [ ]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

In [ ]:
from pyspark.sql.functions import min as _min, max as _max, sum as _sum

category_stats = (
    df.groupBy("category")
    .agg(
        _sum("amount").alias("suma_kwot"),
        _min("amount").alias("min_kwota"),
        _max("amount").alias("max_kwota")
    )
    .orderBy("category")
)
category_stats.show()

In [ ]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

In [ ]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

In [ ]:
store_30min = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .orderBy("window", "store")
)
store_30min.show(truncate=False)

In [ ]:
from pyspark.sql.functions import desc

krakow_hourly = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(_sum("amount"), 2).alias("suma_PLN"))
    .orderBy(desc("suma_PLN"))
)
krakow_hourly.show(1, truncate=False)

In [ ]:
# 1. Gdańsk - godzina z najniższą średnią kwotą
gdansk_min_avg = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(avg("amount"), 2).alias("srednia_kwota"))
    .orderBy("srednia_kwota")
)
print("Godzina z najniższą średnią w Gdańsku:")
gdansk_min_avg.show(1, truncate=False)

# 2. Liczba transakcji per kategoria w oknie 09:00–09:30
cat_0900_0930 = (
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(count("tx_id").alias("liczba_transakcji"))
    .filter(col("window.start") == "2026-04-26 09:00:00")
    .orderBy("category")
)
print("Statystyki kategorii 09:00-09:30:")
cat_0900_0930.show()

# 3. Szczyt transakcji w oknie 15-minutowym (wszystkie sklepy)
peak_15min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("liczba_transakcji"))
    .orderBy(desc("liczba_transakcji"))
)
print("Szczytowa ćwierćgodzina:")
peak_15min.show(1, truncate=False)